In [1]:
import sys
from pathlib import Path

def find_project_root(marker="AGENTS.md"):
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"marker not found from {current}")

root = find_project_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("interpreter:", sys.executable)
print("project root:", root)

import importlib
from src.features import engineering
importlib.reload(engineering)

print("file:", engineering.__file__)
print("has atr_pct:", "atr_pct" in engineering.FEATURE_COLUMNS)

interpreter: /Users/yangjaehoon/Desktop/StockLens/.venv/bin/python
project root: /Users/yangjaehoon/Desktop/StockLens
file: /Users/yangjaehoon/Desktop/StockLens/src/features/engineering.py
has atr_pct: True


In [2]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

def find_project_root(marker="AGENTS.md"):
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"marker not found from {current}")

PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: /Users/yangjaehoon/Desktop/StockLens
RAW_DIR: /Users/yangjaehoon/Desktop/StockLens/data/raw
PROCESSED_DIR: /Users/yangjaehoon/Desktop/StockLens/data/processed


In [3]:
for path in RAW_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))

data/raw/.DS_Store
data/raw/.gitkeep
data/raw/kiwoom/.DS_Store
data/raw/kiwoom/ka10081/.DS_Store
data/raw/kiwoom/ka10081/005380/20260831T051621327442Z.json
data/raw/kiwoom/ka10081/005380/20260916T015006972329Z.json
data/raw/kiwoom/ka10081/005380/20260831T032633950419Z.json
data/raw/kiwoom/ka10081/005380/20260831T032705972234Z.json
data/raw/kiwoom/ka10081/005380/20260901T023509357887Z.json
data/raw/kiwoom/ka10081/035720/20260916T015014375020Z.json
data/raw/kiwoom/ka10081/035720/20260831T032635079878Z.json
data/raw/kiwoom/ka10081/035720/20260831T051621473616Z.json
data/raw/kiwoom/ka10081/035720/20260831T032706052542Z.json
data/raw/kiwoom/ka10081/035720/20260901T023509550392Z.json
data/raw/kiwoom/ka10081/035420/20260831T032706011884Z.json
data/raw/kiwoom/ka10081/035420/20260916T015010434075Z.json
data/raw/kiwoom/ka10081/035420/20260831T051621394346Z.json
data/raw/kiwoom/ka10081/035420/20260901T023509461123Z.json
data/raw/kiwoom/ka10081/035420/20260831T032633987564Z.json
data/raw/kiwoom/ka

In [4]:
import json

sample_file = next(
    (RAW_DIR / "kiwoom" / "ka10081" / "005930").glob("*.json")
)

with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

print(type(sample))
print(sample.keys() if isinstance(sample, dict) else "not a dict")

<class 'dict'>
dict_keys(['stk_cd', 'stk_dt_pole_chart_qry', 'return_code', 'return_msg'])


In [5]:
chart_data = sample["stk_dt_pole_chart_qry"]

print(type(chart_data))
print("rows:", len(chart_data))
print("first row:")
print(chart_data[0])

<class 'list'>
rows: 600
first row:
{'cur_prc': '253750', 'trde_qty': '8308403', 'trde_prica': '2080524', 'dt': '20260831', 'open_pric': '249000', 'high_pric': '255000', 'low_pric': '246000', 'pred_pre': '-3250', 'pred_pre_sig': '5', 'trde_tern_rt': '+0.14'}


In [6]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import engineering
from src.features import target

print("src import OK")
print("engineering:", dir(engineering))
print("target:", dir(target))

src import OK
engineering: ['DailyBar', 'FEATURE_COLUMNS', 'SELECTED_FEATURES', 'Sequence', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'build_features', 'np', 'pd']
target: ['DailyBar', 'Sequence', 'TARGET_COLUMN', 'TARGET_HORIZON', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'annotations', 'build_target', 'pd']


In [7]:
import inspect

print("build_features:")
print(inspect.signature(engineering.build_features))

print("\nbuild_target:")
print(inspect.signature(target.build_target))

print("\nFEATURE_COLUMNS:")
print(engineering.FEATURE_COLUMNS)

print("\nTARGET_COLUMN:")
print(target.TARGET_COLUMN)

print("TARGET_HORIZON:")
print(target.TARGET_HORIZON)

build_features:
(bars: 'Sequence[DailyBar]') -> 'pd.DataFrame'

build_target:
(bars: 'Sequence[DailyBar]', *, horizon: 'int' = 5) -> 'pd.DataFrame'

FEATURE_COLUMNS:
('return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'atr_pct', 'macd_hist_pct', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20')

TARGET_COLUMN:
target_return_5d
TARGET_HORIZON:
5


In [8]:
from src.data.dataset import DailyBar

print(DailyBar)

<class 'src.data.models.DailyBar'>


In [9]:
import json

def load_daily_bars(stock_code: str):
    stock_dir = RAW_DIR / "kiwoom" / "ka10081" / stock_code
    files = sorted(stock_dir.glob("*.json"))

    if not files:
        raise FileNotFoundError(f"No JSON files found: {stock_dir}")

    rows = []

    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        rows.extend(data["stk_dt_pole_chart_qry"])

    # 같은 날짜가 여러 JSON에 있을 수 있으므로 날짜 기준 중복 제거
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset="dt").sort_values("dt").reset_index(drop=True)

    print(stock_code, ":", len(df), "bars")
    print(df["dt"].min(), "~", df["dt"].max())

    return df

In [10]:
raw_df = load_daily_bars("005930")

raw_df.head()

005930 : 10931 bars
19850104 ~ 20260916


,cur_prc,trde_qty,trde_prica,dt,open_pric,high_pric,low_pric,pred_pre,pred_pre_sig,trde_tern_rt
0,129,111765,0,19850104,130,130,129,0,0,0.00
1,128,108497,0,19850105,129,129,128,60,0,0.00
2,129,771895,3,19850107,129,130,128,20,0,0.00
3,127,845098,4,19850108,129,129,127,110,0,0.00
4,123,324837,1,19850109,126,126,122,290,0,0.00


In [11]:
import inspect

print(inspect.signature(DailyBar))

(stock_code: 'str', trade_date: 'date', open_price: 'int', high_price: 'int', low_price: 'int', close_price: 'int', volume: 'int', trade_value_million_krw: 'int', previous_close_change: 'int', previous_close_change_sign: 'int', turnover_rate: 'Decimal') -> None


In [12]:
from datetime import date
from decimal import Decimal

def to_daily_bars(df: pd.DataFrame, stock_code: str) -> list[DailyBar]:
    bars = []

    for row in df.itertuples(index=False):
        bars.append(
            DailyBar(
                stock_code=stock_code,
                trade_date=date(
                    int(str(row.dt)[:4]),
                    int(str(row.dt)[4:6]),
                    int(str(row.dt)[6:8]),
                ),
                open_price=int(row.open_pric),
                high_price=int(row.high_pric),
                low_price=int(row.low_pric),
                close_price=int(row.cur_prc),
                volume=int(row.trde_qty),
                trade_value_million_krw=int(row.trde_prica),
                previous_close_change=int(row.pred_pre),
                previous_close_change_sign=int(row.pred_pre_sig),
                turnover_rate=Decimal(str(row.trde_tern_rt)),
            )
        )

    return bars

In [13]:
bars = to_daily_bars(raw_df, "005930")

print("bars:", len(bars))
print(bars[0])

bars: 10931
DailyBar(stock_code='005930', trade_date=datetime.date(1985, 1, 4), open_price=130, high_price=130, low_price=129, close_price=129, volume=111765, trade_value_million_krw=0, previous_close_change=0, previous_close_change_sign=0, turnover_rate=Decimal('0.00'))


In [14]:
features_df = engineering.build_features(bars)

print("shape:", features_df.shape)
print("columns:")
print(features_df.columns.tolist())

features_df.head()

shape: (10931, 28)
columns:
['trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'atr_pct', 'macd_hist_pct', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']


,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,sma_20,...,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,atr_pct,macd_hist_pct,volume_change_1d,volume_sma_20,volume_ratio_20
0,1985-01-04,NaN,NaN,NaN,NaN,-0.007692,0.007752,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1985-01-05,-0.007752,NaN,NaN,NaN,-0.007752,0.007812,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.029240,NaN,NaN
2,1985-01-07,0.007812,NaN,NaN,NaN,0.000000,0.015625,0.007812,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.114436,NaN,NaN
3,1985-01-08,-0.015504,NaN,NaN,NaN,-0.015504,0.015748,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.094835,NaN,NaN
4,1985-01-09,-0.031496,NaN,NaN,NaN,-0.023810,0.032787,-0.007874,127.2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.615622,NaN,NaN


In [15]:
target_df = target.build_target(
    bars,
    horizon=target.TARGET_HORIZON,
)

print("shape:", target_df.shape)
print("columns:", target_df.columns.tolist())

target_df.head(10)

shape: (10931, 3)
columns: ['stock_code', 'trade_date', 'target_return_5d']


,stock_code,trade_date,target_return_5d
0,005930,1985-01-04,-0.038760
1,005930,1985-01-05,-0.023438
2,005930,1985-01-07,-0.015504
3,005930,1985-01-08,-0.007874
4,005930,1985-01-09,0.032520
5,005930,1985-01-10,0.008065
6,005930,1985-01-11,-0.008000
7,005930,1985-01-12,-0.023622
8,005930,1985-01-14,-0.015873
9,005930,1985-01-15,-0.023622


In [16]:
prepared_df = features_df.merge(
    target_df,
    on="trade_date",
    how="inner",
)

print("shape:", prepared_df.shape)
print("columns:", prepared_df.columns.tolist())

prepared_df.head()

shape: (10931, 30)
columns: ['trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'atr_pct', 'macd_hist_pct', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'stock_code', 'target_return_5d']


,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,sma_20,...,volatility_5,volatility_20,atr_14,atr_pct,macd_hist_pct,volume_change_1d,volume_sma_20,volume_ratio_20,stock_code,target_return_5d
0,1985-01-04,NaN,NaN,NaN,NaN,-0.007692,0.007752,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,005930,-0.038760
1,1985-01-05,-0.007752,NaN,NaN,NaN,-0.007752,0.007812,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.029240,NaN,NaN,005930,-0.023438
2,1985-01-07,0.007812,NaN,NaN,NaN,0.000000,0.015625,0.007812,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,6.114436,NaN,NaN,005930,-0.015504
3,1985-01-08,-0.015504,NaN,NaN,NaN,-0.015504,0.015748,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.094835,NaN,NaN,005930,-0.007874
4,1985-01-09,-0.031496,NaN,NaN,NaN,-0.023810,0.032787,-0.007874,127.2,NaN,...,NaN,NaN,NaN,NaN,NaN,-0.615622,NaN,NaN,005930,0.032520


In [17]:
missing = prepared_df.isna().sum()

missing[missing > 0]

return_1d            1
return_5d            5
return_10d          10
return_20d          20
gap                  1
sma_5                4
sma_20              19
sma_60              59
price_to_sma_5       4
price_to_sma_20     19
price_to_sma_60     59
rsi_14              14
roc_10              10
roc_20              20
macd                25
macd_signal         33
macd_hist           33
volatility_5         5
volatility_20       20
atr_14              13
atr_pct             13
macd_hist_pct       33
volume_change_1d     3
volume_sma_20       19
volume_ratio_20     19
target_return_5d     5
dtype: int64

In [18]:
print("전체 행:", len(prepared_df))
print("결측치가 하나라도 있는 행:", prepared_df.isna().any(axis=1).sum())

전체 행: 10931
결측치가 하나라도 있는 행: 66


In [19]:
clean_df = prepared_df.dropna(
    subset=list(engineering.FEATURE_COLUMNS) + [target.TARGET_COLUMN]
).reset_index(drop=True)

print("Before:", prepared_df.shape)
print("After :", clean_df.shape)
print("Removed:", len(prepared_df) - len(clean_df))

print("\nRemaining missing values:")
print(clean_df.isna().sum().sum())

Before: (10931, 30)
After : (10865, 30)
Removed: 66

Remaining missing values:
0


In [20]:
print("첫 70행의 결측치:")
print(
    prepared_df[list(engineering.FEATURE_COLUMNS) + [target.TARGET_COLUMN]]
    .isna()
    .sum(axis=1)
    .head(70)
)

첫 70행의 결측치:
0     25
1     22
2     22
3     22
4     20
      ..
65     0
66     0
67     0
68     0
69     0
Length: 70, dtype: int64


In [21]:
print("shape:", clean_df.shape)
print("missing:", clean_df.isna().sum().sum())

print("\nDate range:")
print(clean_df["trade_date"].min(), "~", clean_df["trade_date"].max())

shape: (10865, 30)
missing: 0

Date range:
1985-03-18 ~ 2026-09-09


In [22]:
print("=== Dataset columns ===")
print(clean_df.columns.tolist())

print("\n=== Date range ===")
print(clean_df["trade_date"].min(), "~", clean_df["trade_date"].max())

print("\n=== Target range ===")
print(clean_df["target_return_5d"].min(), "~", clean_df["target_return_5d"].max())

=== Dataset columns ===
['trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'atr_pct', 'macd_hist_pct', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'stock_code', 'target_return_5d']

=== Date range ===
1985-03-18 ~ 2026-09-09

=== Target range ===
-0.29593495934959346 ~ 0.37451737451737444


In [23]:
clean_df.tail(10)[
    ["trade_date", "target_return_5d"]
]

,trade_date,target_return_5d
10855,2026-08-27,-0.060150
10856,2026-08-28,-0.005837
10857,2026-08-31,0.071429
10858,2026-09-01,0.041546
10859,2026-09-02,0.075848
10860,2026-09-03,0.076000
10861,2026-09-04,0.015656
10862,2026-09-07,-0.079630
10863,2026-09-08,-0.070501
10864,2026-09-09,-0.070501


In [24]:
# Feature가 현재 시점 이후의 데이터를 사용하지 않는지 확인하기 위한
# 가장 기본적인 시점 검증

print("Feature columns:")
print(list(engineering.FEATURE_COLUMNS))

print("\nClean dataset:")
print(clean_df.shape)

print("\nDate range:")
print(clean_df["trade_date"].min(), "~", clean_df["trade_date"].max())

Feature columns:
['return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'atr_pct', 'macd_hist_pct', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20']

Clean dataset:
(10865, 30)

Date range:
1985-03-18 ~ 2026-09-09


In [25]:
final_columns = [
    "stock_code",
    "trade_date",
    *engineering.FEATURE_COLUMNS,
    target.TARGET_COLUMN,
]

final_df = clean_df[final_columns].copy()

print(final_df.shape)
print(final_df.columns.tolist())

(10865, 30)
['stock_code', 'trade_date', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'atr_pct', 'macd_hist_pct', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'target_return_5d']


In [26]:
STOCK_CODES = [
    "000660",
    "005380",
    "005930",
    "035420",
    "035720",
]

print(STOCK_CODES)

['000660', '005380', '005930', '035420', '035720']


In [27]:
import json
from datetime import date
from decimal import Decimal

from src.data.models import DailyBar


def load_bars(stock_code):
    stock_dir = RAW_DIR / "kiwoom" / "ka10081" / stock_code

    rows_by_date = {}

    for json_path in sorted(stock_dir.glob("*.json")):
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        rows = data.get("stk_dt_pole_chart_qry", [])

        for row in rows:
            dt = row["dt"]

            rows_by_date[dt] = DailyBar(
                stock_code=stock_code,
                trade_date=date(
                    int(dt[:4]),
                    int(dt[4:6]),
                    int(dt[6:8]),
                ),
                open_price=int(row["open_pric"]),
                high_price=int(row["high_pric"]),
                low_price=int(row["low_pric"]),
                close_price=int(row["cur_prc"]),
                volume=int(row["trde_qty"]),
                trade_value_million_krw=int(row["trde_prica"]),
                previous_close_change=int(row["pred_pre"]),
                previous_close_change_sign=int(row["pred_pre_sig"]),
                turnover_rate=Decimal(row["trde_tern_rt"]),
            )

    return sorted(rows_by_date.values(), key=lambda x: x.trade_date)

In [28]:
prepared_dfs = []

for stock_code in STOCK_CODES:
    bars = load_bars(stock_code)

    features = engineering.build_features(bars)
    targets = target.build_target(
        bars,
        horizon=target.TARGET_HORIZON,
    )

    df = features.merge(
        targets,
        on="trade_date",
        how="inner",
    )

    df["stock_code"] = stock_code

    df = df[
        [
            "stock_code",
            "trade_date",
            *engineering.FEATURE_COLUMNS,
            target.TARGET_COLUMN,
        ]
    ]

    df = df.dropna().reset_index(drop=True)

    prepared_dfs.append(df)

    print(
        stock_code,
        "bars:", len(bars),
        "prepared:", df.shape,
    )

prepared_df = pd.concat(
    prepared_dfs,
    ignore_index=True,
)

print("\nTOTAL:", prepared_df.shape)

000660 bars: 7418 prepared: (7343, 30)
005380 bars: 10931 prepared: (10865, 30)
005930 bars: 10931 prepared: (10865, 30)
035420 bars: 5895 prepared: (5809, 30)
035720 bars: 6617 prepared: (6538, 30)

TOTAL: (41420, 30)


In [29]:
output_path = PROCESSED_DIR / "ml_dataset.csv"

prepared_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", prepared_df.shape)

Saved: /Users/yangjaehoon/Desktop/StockLens/data/processed/ml_dataset.csv
Shape: (41420, 30)


In [30]:
import pandas as pd

from src.data.dataset import (
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    VALIDATION_START_DATE,
    VALIDATION_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
)

# ============================================
# Train / Validation / Test Time Split
# ============================================

# 이미 전처리된 전체 ML dataset 불러오기
ml_dataset_path = PROCESSED_DIR / "ml_dataset.csv"
ml_dataset = pd.read_csv(ml_dataset_path)

# 날짜 타입 변환
ml_dataset["trade_date"] = pd.to_datetime(ml_dataset["trade_date"])

# 공식 시간 분할 기준 -- src.data.dataset이 유일한 기준(single source of
# truth). 여기서 날짜를 따로 하드코딩하면 소스가 바뀌어도 노트북이 반영을
#못 하니, 절대 따로 적지 말고 항상 여기서 import해서 쓸 것.
TRAIN_START = TRAIN_START_DATE
TRAIN_END = TRAIN_END_DATE

VALIDATION_START = VALIDATION_START_DATE
VALIDATION_END = VALIDATION_END_DATE

TEST_START = TEST_START_DATE
TEST_END = TEST_END_DATE

# Time-based split
train_df = ml_dataset[
    (ml_dataset["trade_date"] >= TRAIN_START) &
    (ml_dataset["trade_date"] <= TRAIN_END)
].copy()

validation_df = ml_dataset[
    (ml_dataset["trade_date"] >= VALIDATION_START) &
    (ml_dataset["trade_date"] <= VALIDATION_END)
].copy()

test_df = ml_dataset[
    (ml_dataset["trade_date"] >= TEST_START) &
    (ml_dataset["trade_date"] <= TEST_END)
].copy()

# 날짜순 + 종목순 정렬
train_df = train_df.sort_values(
    ["trade_date", "stock_code"]
).reset_index(drop=True)

validation_df = validation_df.sort_values(
    ["trade_date", "stock_code"]
).reset_index(drop=True)

test_df = test_df.sort_values(
    ["trade_date", "stock_code"]
).reset_index(drop=True)

# CSV 저장
train_df.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

validation_df.to_csv(
    PROCESSED_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)

# ============================================
# 결과 확인
# ============================================

print("=== Time Split Result ===")
print(f"Train      : {train_df.shape}")
print(f"Validation : {validation_df.shape}")
print(f"Test       : {test_df.shape}")

print("\n=== Date Range ===")
print(
    f"Train      : "
    f"{train_df['trade_date'].min().date()} ~ "
    f"{train_df['trade_date'].max().date()}"
)
print(
    f"Validation : "
    f"{validation_df['trade_date'].min().date()} ~ "
    f"{validation_df['trade_date'].max().date()}"
)
print(
    f"Test       : "
    f"{test_df['trade_date'].min().date()} ~ "
    f"{test_df['trade_date'].max().date()}"
)

print("\n=== Number of Stocks ===")
print(f"Train      : {train_df['stock_code'].nunique()}")
print(f"Validation : {validation_df['stock_code'].nunique()}")
print(f"Test       : {test_df['stock_code'].nunique()}")


=== Time Split Result ===
Train      : (21133, 30)
Validation : (4323, 30)
Test       : (3885, 30)

=== Date Range ===
Train      : 2002-10-29 ~ 2019-12-30
Validation : 2020-01-02 ~ 2023-06-30
Test       : 2023-07-03 ~ 2026-09-09

=== Number of Stocks ===
Train      : 5
Validation : 5
Test       : 5
